In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from scipy.stats import mannwhitneyu

# ---------------------------------------------------------
# 1. Load the two Excel files
# ---------------------------------------------------------
df_piad = pd.read_excel('PiAD_Actin_Data.xlsx')
df_phys = pd.read_excel('Trypsin_Actin_Data.xlsx')

# Automatically determine frame interval (dt) from each file
dt_piad = df_piad['Time_s'].diff().mean()
dt_phys = df_phys['Time_s'].diff().mean()

print(f"PiAD frame interval: {dt_piad:.2f} s")
print(f"Physiological frame interval: {dt_phys:.2f} s")

# ---------------------------------------------------------
# 2. Analysis function
# ---------------------------------------------------------
def analyze_bleb_kinetics(df, dt, group_label, window_len=7, polyorder=2):
    results = []
    roi_cols = [c for c in df.columns if c != 'Time_s']

    for col in roi_cols:
        raw_signal = df[col].dropna().values
        time = df['Time_s'].values[:len(raw_signal)]

        # Smoothing with filter
        w_len = min(window_len, len(raw_signal) if len(raw_signal) % 2 != 0 else len(raw_signal) - 1)
        smooth_signal = savgol_filter(raw_signal, window_length=w_len, polyorder=polyorder)

        # Baseline normalization (F / F0)
        f0 = np.mean(smooth_signal[:5])
        norm_signal = smooth_signal / f0

        # Time derivative (d[Actin]/dt)
        d_actin_dt = np.gradient(norm_signal, dt)

        # ① Maximum Recovery Rate (d[Actin]/dt max)
        max_rate = np.max(d_actin_dt)

        # ④ Maximum Intensity / Plateau Level
        max_intensity = np.max(norm_signal)
        min_intensity = np.min(norm_signal[:10])

        # ② Peak Time & Half-time
        peak_idx = np.argmax(norm_signal)
        t_peak = time[peak_idx]

        bleb_start_idx = np.argmin(norm_signal[:peak_idx+1])
        t_start = time[bleb_start_idx]

        half_intensity = min_intensity + (max_intensity - min_intensity) / 2.0
        recovery_phase_signal = norm_signal[bleb_start_idx:peak_idx+1]
        recovery_phase_time = time[bleb_start_idx:peak_idx+1]

        half_idx = np.argmin(np.abs(recovery_phase_signal - half_intensity))
        t_half = recovery_phase_time[half_idx] - t_start

        # ③ Bleb Lifetime
        after_peak_signal = norm_signal[peak_idx:]
        after_peak_time = time[peak_idx:]

        retract_threshold = min_intensity + 0.15 * (max_intensity - min_intensity)
        below_thresh = np.where(after_peak_signal <= retract_threshold)[0]

        t_end = after_peak_time[below_thresh[0]] if len(below_thresh) > 0 else time[-1]
        bleb_lifetime = t_end - t_start

        results.append({
            'Group': group_label,
            'ROI': col,
            'Max_Recovery_Rate': max_rate,
            't_half_s': t_half,
            'Peak_Time_s': t_peak - t_start,
            'Bleb_Lifetime_s': bleb_lifetime,
            'Max_Plateau_Intensity': max_intensity
        })

    return pd.DataFrame(results)

df_piad_res = analyze_bleb_kinetics(df_piad, dt_piad, 'PiAD')
df_phys_res = analyze_bleb_kinetics(df_phys, dt_phys, 'Physiological')

# ---------------------------------------------------------
# 3. Integration and U-test results output
# ---------------------------------------------------------
metrics = ['Max_Recovery_Rate', 't_half_s', 'Bleb_Lifetime_s', 'Max_Plateau_Intensity']
metric_names = [
    '① Max Recovery Rate (ΔF/F0/s)',
    '② Half-time Recovery t1/2 (s)',
    '③ Bleb Lifetime (s)',
    '④ Max Plateau Intensity (F/F0)'
]

def get_median_iqr(series):
    med = series.median()
    q25 = series.quantile(0.25)
    q75 = series.quantile(0.75)
    return f"{med:.3f} [{q25:.3f}-{q75:.3f}]"

stat_summary = []
for m, name in zip(metrics, metric_names):
    piad_vals = df_piad_res[m]
    phys_vals = df_phys_res[m]

    stat, p_val = mannwhitneyu(piad_vals, phys_vals, alternative='two-sided')

    stat_summary.append({
        'Metric': name,
        'PiAD Median [IQR]': get_median_iqr(piad_vals),
        'Physiological Median [IQR]': get_median_iqr(phys_vals),
        'U-statistic': f"{stat:.1f}",
        'p-value': f"{p_val:.4f}",
        'Significance': 'n.s. (Not Significant)' if p_val > 0.05 else 'p < 0.05'
    })

stat_df = pd.DataFrame(stat_summary)
print("\n=== Mann-Whitney U Test Results ===")
print(stat_df.to_string(index=False))

# ---------------------------------------------------------
# 4. Plotting graphs (Y-axis minimum set to 0)
# ---------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(6, 5))

# Function to generate horizontal jitter for data points
def add_jitter(data, x_pos, jitter_val=0.08):
    np.random.seed(42)
    return x_pos + np.random.normal(0, jitter_val, size=len(data))

# Common boxplot style configuration (line width 1.5)
box_style_A = dict(
    boxprops=dict(facecolor='#E6B0AA', color='#D9534F', alpha=0.6, linewidth=2.0),
    whiskerprops=dict(color='#D9534F', linewidth=2.0),
    capprops=dict(color='#D9534F', linewidth=1.5),
    medianprops=dict(color='black', linewidth=2.5),
    showfliers=False
)

box_style_B = dict(
    boxprops=dict(facecolor='#D5F5E3', color='#2ECC71', alpha=0.6, linewidth=2.0),
    whiskerprops=dict(color='#2ECC71', linewidth=2.0),
    capprops=dict(color='#2ECC71', linewidth=1.5),
    medianprops=dict(color='black', linewidth=2.5),
    showfliers=False
)

# A: Maximum Recovery Rate
piad_rate = df_piad_res['Max_Recovery_Rate']
phys_rate = df_phys_res['Max_Recovery_Rate']

axes[0].boxplot([piad_rate, phys_rate], labels=['PiAD', 'Physiological'], widths=0.5, patch_artist=True, **box_style_A)
axes[0].scatter(add_jitter(piad_rate, 1), piad_rate, color='#D9534F', alpha=0.7, edgecolors='none', s=25)
axes[0].scatter(add_jitter(phys_rate, 2), phys_rate, color='#428BCA', alpha=0.7, edgecolors='none', s=25)

axes[0].set_ylabel('d[Actin]/dt max (ΔF/F0 / s)', fontsize=10, fontweight='bold')
axes[0].set_title('A. Maximum Recovery Rate', fontsize=10, fontweight='bold')

# B: Max Plateau Intensity
piad_int = df_piad_res['Max_Plateau_Intensity']
phys_int = df_phys_res['Max_Plateau_Intensity']

axes[1].boxplot([piad_int, phys_int], labels=['PiAD', 'Physiological'], widths=0.5, patch_artist=True, **box_style_B)
axes[1].scatter(add_jitter(piad_int, 1), piad_int, color='#2ECC71', alpha=0.7, edgecolors='none', s=25)
axes[1].scatter(add_jitter(phys_int, 2), phys_int, color='#428BCA', alpha=0.7, edgecolors='none', s=25)

axes[1].set_ylabel('Max Intensity (F/F0)', fontsize=10, fontweight='bold')
axes[1].set_title('B. Max Plateau Intensity', fontsize=10, fontweight='bold')

# Customize axes and borders (common for both panels)
for ax in axes:
    # Remove horizontal grid lines
    ax.grid(False)

    # Remove top and right spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Set bottom and left spines thicker
    ax.spines['left'].set_linewidth(2.5)
    ax.spines['bottom'].set_linewidth(2.5)

    # Adjust tick line thickness
    ax.tick_params(axis='both', width=2.5, labelsize=9)

    # ★ Set Y-axis minimum to 0 (upper limit is automatically calculated based on data)
    ax.set_ylim(bottom=0)

plt.tight_layout()
plt.show()